## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 54.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-exp

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.3 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
# Model names
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [5]:
# Model path
model_path = hf_hub_download(
  repo_id = model_name_or_path,
  filename = model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
# Llama
llm = Llama(
  model_path = model_path,
  n_ctx = 2300,
  n_gpu_layers = 38,
  n_batch = 512
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [7]:
# response function definition
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt = query,
      max_tokens = max_tokens,
      temperature = temperature,
      top_p = top_p,
      top_k = top_k
    )

    return model_output['choices'][0]['text']

In [8]:
# The response function in action
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [9]:
# The first query definition and user input
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Source control'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [10]:
# The second query definition and user input
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right quadrant of the abdomen and may start as a mild discomfort that gradually worsens over time. The pain may be constant or intermittent and may be aggravated by movement, deep breathing, or coughing.\n2. Loss of appetite'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [11]:
# The third query definition and user input
user_input_2 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_2)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [12]:
# The fourth query definition and user input
user_input_2 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_2)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:\n\n1. Emergency care: The first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating normally. In severe cases, emergency surgery may be required to remove hematomas or other obstructions.\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as"

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [13]:
# The fifth query definition and user input
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_2)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or low blood pressure. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint,"

## Question Answering using LLM with Prompt Engineering

In [14]:
# The system prompt
system_prompt = "You are a nurse working in the ER of a hospital."

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [15]:
# The first query pass and user input
user_input = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that requires prompt recognition and intervention. In a critical care unit, managing sepsis involves a multidisciplinary approach and adherence to evidence-based guidelines. Here are the general steps for managing sepsis in a critical care unit:\n\n1. Early recognition and assessment: Recognize the signs and symptoms of sepsis, including fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis. Use the Sequential Organ Failure Ass'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [16]:
# The second query pass and user input
user_input = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms of appendicitis:\n\n1. Abdominal pain: The pain usually starts as a mild discomfort around the navel that may come and go. However, it soon localizes to the lower right side of the abdomen and becomes constant and severe. The pain may be sharp or dull and worsens with movement, coughing, or sneezing.\n2. Loss of appetite: People with'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [17]:
# The third query pass and user input
user_input = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that causes hair loss in small patches on the scalp or other parts of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system.\n\nThere are currently no definitive cures for alopecia areata, but there are several treatments that can help stimulate hair growth and improve the appearance of bald spots:\n\n1. Corticosteroids: These medications are often used to treat"

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [18]:
# The fourth query pass and user input
user_input = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), can experience a wide range of symptoms depending on the severity and location of the injury. Treatment for TBIs typically involves a combination of interventions aimed at managing acute symptoms, preventing complications, and promoting recovery. Here are some common treatments recommended for individuals with TBIs:\n\n1. Emergency care: The first priority in treating a person with a TBI is to ensure their airway is clear, they are breathing adequately, and their circulation is stable. This'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [19]:
# The fifth query pass and user input
user_input = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nWhen a patient presents to the ER with a suspected leg fracture after a hiking accident, it is essential to follow specific procedures to ensure their safety, proper diagnosis, and effective treatment. Here are the necessary precautions and treatment steps:\n\n1. Assess the patient's vital signs and overall condition: Check the patient's pulse, respiratory rate, blood pressure, and temperature. Evaluate their level of consciousness, pain level, and any signs of shock or bleeding.\n2. Immobilize the fractured leg: Apply a splint or a cast to immobilize the leg and"

## Data Preparation for RAG

### Loading the Data

In [20]:
# Connecting Google Colab and Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# The manual's path
manual_pdf_path = "/content/drive/MyDrive/medical_diagnosis_manual.pdf"

In [22]:
# Loading the PDF
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [23]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [24]:
for i in range(5):
  print(f"Page Number : {i+1}", end = "\n")
  print(manual[i].page_content, end = "\n")

Page Number : 1
ginny.dole@icloud.com
UN6L5G2X18
This file is meant for personal use by ginny.dole@icloud.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
ginny.dole@icloud.com
UN6L5G2X18
This file is meant for personal use by ginny.dole@icloud.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...............................................................................................................................

#### Checking the number of pages

In [25]:
len(manual)

4114

### Data Chunking

In [26]:
# The text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
  encoding_name = 'cl100k_base',
  chunk_size = 500,
  chunk_overlap = 100
)

In [27]:
# Document chunks
document_chunks = pdf_loader.load_and_split(text_splitter)

In [28]:
# Length of document chunks
len(document_chunks)

9372

In [29]:
# Content of first page
document_chunks[0].page_content

'ginny.dole@icloud.com\nUN6L5G2X18\nThis file is meant for personal use by ginny.dole@icloud.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [30]:
# Content of second page
document_chunks[1].page_content

'ginny.dole@icloud.com\nUN6L5G2X18\nThis file is meant for personal use by ginny.dole@icloud.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [31]:
# Content of third page
document_chunks[2].page_content

'Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [32]:
# Content of fourth page
document_chunks[3].page_content

"246\nChapter 21. Anorectal Disorders    ............................................................................................................................................\n254\nChapter 22. Tumors of the GI Tract    ......................................................................................................................................\n275\n3 - Hepatic & Biliary Disorders    ............................................................................................................................................\n275\nChapter 23. Approach to the Patient With Liver Disease    ...........................................................................................\n294\nChapter 24. Testing for Hepatic & Biliary Disorders    ......................................................................................................\n305\nChapter 25. Drugs & the Liver    ..................................................................................................

### Embedding

In [33]:
# Selecting embedding model
embedding_model = SentenceTransformerEmbeddings(model_name = "all-MiniLM-L6-v2")

/tmp/ipykernel_55566/3044234237.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name = "all-MiniLM-L6-v2")


In [34]:
# Embedding 1 and 2
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [35]:
# Are the embedding lengths the same?
print("Dimension of the embedding vector ", len(embedding_1))
len(embedding_1) == len(embedding_2)

Dimension of the embedding vector  384


True

In [36]:
embedding_1, embedding_2

([-0.09891160577535629,
  0.057196326553821564,
  0.07006581127643585,
  -0.05857103690505028,
  0.06484948098659515,
  -0.02782977931201458,
  -0.012267041951417923,
  0.024499565362930298,
  0.11176405847072601,
  0.06559988856315613,
  -0.009089360944926739,
  0.12315068393945694,
  -0.0260403323918581,
  -0.02542172186076641,
  -0.03822366148233414,
  0.028730887919664383,
  -0.07825854420661926,
  0.05205600708723068,
  -0.10944826900959015,
  0.048758164048194885,
  0.04767022281885147,
  0.05832777917385101,
  0.05893906578421593,
  -0.030888907611370087,
  0.03238442912697792,
  -0.06227794289588928,
  -0.060740042477846146,
  0.012270431965589523,
  -0.10419337451457977,
  0.04005735367536545,
  0.04600362479686737,
  0.09684973955154419,
  -0.02403944730758667,
  0.022025832906365395,
  0.0076577505096793175,
  -0.009587046690285206,
  0.04113247990608215,
  -0.014276769012212753,
  0.0007250701892189682,
  0.0297537948936224,
  -0.009787357412278652,
  -0.01624893955886364,


### Vector Database

In [37]:
# Making a directory
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [38]:
# Storing vectors with document chunks and embedding model
vectorstore = Chroma.from_documents(
  documents = document_chunks,
  embedding = embedding_model,
  persist_directory = out_dir
)

In [39]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function = embedding_model)

/tmp/ipykernel_55566/1704797186.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function = embedding_model)


In [40]:
# Embeddings
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [41]:
# Similarity search
vectorstore.similarity_search("What is the protocol for managing sepsis in a critical care unit?", k = 5)

[Document(metadata={'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'author': '', 'creator': 'Atop CHM to PDF Converter', 'subject': '', 'keywords': '', 'total_pages': 4114, 'creationDate': 'D:20120615054440Z', 'moddate': '2026-05-30T00:13:33+00:00', 'creationdate': '2012-06-15T05:44:40+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'trapped': '', 'page': 2400, 'modDate': 'D:20260530001333Z', 'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs ha

### Retriever

In [42]:
# The retriever
retriever = vectorstore.as_retriever(
  search_type = 'similarity',
  search_kwargs = {'k': 5}
)

In [43]:
# Relevant documents
rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")
rel_docs

/tmp/ipykernel_55566/1556909376.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")


[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'total_pages': 4114, 'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'moddate': '2026-05-30T00:13:33+00:00', 'format': 'PDF 1.7', 'trapped': '', 'subject': '', 'author': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationdate': '2012-06-15T05:44:40+00:00', 'page': 2400, 'keywords': '', 'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'modDate': 'D:20260530001333Z'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs ha

In [44]:
# The model output
model_output = llm(
  "What is the protocol for managing sepsis in a critical care unit?",
  max_tokens = 450,
  temperature = 0,
)

Llama.generate: prefix-match hit


In [45]:
# The choices
model_output['choices'][0]['text']

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Source control: Identify and address the source of infection, if possible. This may involve surgical intervention, such as drainage of an abscess or debridement of necrotic tissue.\n3. Fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and organ perfusion. The goal is to achieve a mean arterial pressure (MAP) of at least 65 mmHg and a central venous oxygen saturation (ScvO2) of greater than 70%.\n4. Vasopressor support: 

### System and User Prompt Template

In [46]:
# The Q&A system message
qna_system_message = """
  You answer questions only using info from  ###Context.
  Do not add content beyond this.
  If not possible, reply with, "I don't know."
  Do not mention context or question tokens in the answer.
"""

In [47]:
# The Q&A user message
qna_user_message_template = """
Consider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

### Response Function

In [48]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
  global qna_system_message,qna_user_message_template
  # Retrieve relevant document chunks
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
  context_list = [d.page_content for d in relevant_document_chunks]

  # Combine document chunks into a single context
  context_for_query = ". ".join(context_list)

  user_message = qna_user_message_template.replace('{context}', context_for_query)
  user_message = user_message.replace('{question}', user_input)

  prompt = qna_system_message + '\n' + user_message

  # Generate the response
  try:
    response = llm(
      prompt=prompt,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    # Extract and print the model's response
    response = response['choices'][0]['text'].strip()
  except Exception as e:
    response = f'Sorry, I encountered the following error: \n {e}'

  return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [49]:
# The first query pass and user input
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input, top_k = 20)

Llama.generate: prefix-match hit


'The protocol for managing sepsis in a critical care unit includes obtaining cultures of blood and any other appropriate specimens if bacteremia, sepsis, or septic shock is suspected. Early treatment with an appropriate antimicrobial regimen improves survival. Continuing therapy involves adjusting antibiotics according to culture and susceptibility testing, surgically draining any abscesses, and removing internal devices that are the suspected source of bacteria.'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [50]:
# The second query pass and user input
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2, top_k = 20)

Llama.generate: prefix-match hit


"The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, anorexia, abdominal tenderness, right lower quadrant pain that increases with cough and motion, direct and rebound tenderness located at McBurney's point, Rovsing sign (pain felt in the right lower quadrant with palpation of the left lower quadrant), psoas sign (pain caused by passive internal rotation of the flexed thigh), obturator sign (pain caused by passive extension of the hip joint), and"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [51]:
# The third query pass and user input
user_input_2 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_2, top_k = 20)

Llama.generate: prefix-match hit


'Based on the context provided, the condition described in the question is likely Alopecia Areata. The effective treatments or solutions mentioned in the context for this condition include:\n1. Long-acting oral tetracycline in combination with a potent topical corticosteroid for scarring alopecias such as central centrifugal scarring alopecia, dissecting cellulitis of the scalp, and acne keloidalis nuchae.\n2. Oral antimalarials, corticosteroids, retinoids, or immunosupp'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [52]:
# The fourth query pass and user input
user_input_2 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_2, top_k = 20)

Llama.generate: prefix-match hit


'The Merck Manual recommends early intervention by rehabilitation specialists for maximal functional recovery from a brain injury. Treatments include prevention of secondary disabilities such as pressure ulcers and joint contractures, prevention of pneumonia, and family education. Patients should be evaluated to establish baseline findings before starting rehabilitation therapy, with findings compared to help prioritize treatment. For severe cognitive dysfunction, extensive cognitive therapy is often begun immediately after injury and continued for months or years. Physical and occupational therapy may modestly improve functioning but is more often useful for making the environment safer and providing devices to help patients circumvent primary'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [53]:
# The fifth query pass and user input
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_2, top_k = 20)

Llama.generate: prefix-match hit


'The context discusses preventive measures and treatments for Deep Vein Thrombosis (DVT) and injuries. For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:\n\n1. Rest: The person should avoid putting weight on the injured leg to prevent further injury and promote healing.\n2. Ice: Applying ice intermittently for 15 to 20 minutes as often as possible can help minimize swelling and pain.\n3. Compression: Wrapping the injured leg with an elastic bandage or using a spl'

### Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [54]:
# The first query pass and user input
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input, temperature = 0.5)

Llama.generate: prefix-match hit


'Based on the context provided, the protocol for managing sepsis in a critical care unit includes:\n1. Suspecting sepsis or septic shock based on symptoms such as shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea).\n2. Obtaining cultures of blood and any other appropriate specimens for diagnosis.\n3. Administering empiric antibiotics after obtaining cultures.\n4. Continuing therapy by adjusting antibiotics according to culture and susceptibility'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [55]:
# The second query pass and user input
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2, temperature = 0.5)

Llama.generate: prefix-match hit


"Based on the context, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. However, these symptoms may not always be present in all patients, especially in infants, children, elderly patients, and pregnant women. Appendicitis cannot be cured via medicine alone, as it requires surgical"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [56]:
# The third query pass and user input
user_input_2 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_2, temperature = 0.5)

Llama.generate: prefix-match hit


'The most common cause of sudden patchy hair loss mentioned in the context is Alopecia Areata. The context states that it is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. The treatments or solutions suggested for addressing this condition include:\n- Microscopic hair examination or scalp biopsy for definitive diagnosis.\n- Oral antimalarials, corticosteroids, retinoids, or immunosuppressants as potential treatment options.\nThe context also mentions that alopecia areata may affect any hairy area of'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [57]:
# The fourth query pass and user input
user_input_2 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_2, temperature = 0.5)

Llama.generate: prefix-match hit


'Based on the context provided, the following treatments are recommended for a person who has sustained a physical injury to brain tissue:\n\n1. Early intervention by rehabilitation specialists: This is crucial for maximal functional recovery and includes prevention of secondary disabilities, such as pressure ulcers, joint contractures, and pneumonia. Rehabilitation specialists should evaluate patients as early as possible to establish baseline findings and prioritize treatment based on subsequent evaluations.\n2. Cognitive therapy: Patients with severe cognitive dysfunction require extensive cognitive therapy, which is often begun immediately after injury and continued for months or years.\n3'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [58]:
# The fifth query pass and user input
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_2, temperature = 0.5)

Llama.generate: prefix-match hit


'The context provides information about various preventive measures and treatments for Deep Vein Thrombosis (DVT) and injuries. The person in question has fractured their leg, which is not directly related to DVT but rather to an injury. Therefore, the necessary precautions and treatment steps for a person with a fractured leg would be as follows:\n\n1. Immobilization: The injured leg should be immobilized immediately using a splint or a cast to prevent further injury and decrease pain. This is mentioned in the context as a common practice for most injuries, particularly those that are grossly unstable'

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [59]:
groundedness_rater_system_message = """
You will rate AI-generated answers based on how well they follow the rule, "The answer must be derived only from the provided context."

Input format:
-Question begins with ###Question
-Context begins with ###Context
-Answer begins with ###Answer

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
RATING: [number] (1-5)

Metric:
The answer must be fully based on the provided context.
Do not include any details, claims, or assumptions not explicitly supported by the context.

Instructions:
1. List top 3 steps to check if answer is fully supported by the metric.
2. Explain in 1-2 sentences if the answer is fully grounded in the given context.
3. Evaluate the extent to which the metric is followed.
4. Based on the assessment, assign a rating using the evaluation criteria.
"""

In [60]:
relevance_rater_system_message = """
You will rate AI-generated answers based on how well they follow the rule, "The answer must be derived only from the provided context."

Input format:
-Question begins with ###Question
-Context begins with ###Context
-Answer begins with ###Answer

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
RATING: [number] (1-5)

Metric:
The answer must be fully based on the provided context.
Do not include any details, claims, or assumptions not explicitly supported by the context.

Instructions:
1. List top 3 steps to check if answer is fully supported by the metric.
2. Explain in 1-2 sentences if the answer is fully grounded in the given context.
3. Evaluate the extent to which the metric is followed.
4. Based on the assessment, assign a rating using the evaluation criteria.
"""

In [61]:
# The user message template
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [62]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [63]:
# The first query pass and user input
ground,rel = generate_ground_relevance_response(user_input = "What is the protocol for managing sepsis in a critical care unit?", max_tokens = 370)

print(ground,end = "\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Top 3 steps to check if answer is fully supported by the metric:
1. Identify specific information in the context related to managing sepsis in a critical care unit.
2. Check if the answer directly quotes or paraphrases this information.
3. Evaluate if any additional details, claims, or assumptions are included in the answer that are not explicitly supported by the context.

Explanation: The answer is fully grounded in the given context as it directly quotes and summarizes the steps for managing sepsis mentioned in the context.

Evaluation: Based on the assessment, I would assign a rating of 5 for this answer as it strictly follows the metric by providing an answer that is fully based on the provided context without any additional details or assumptions.

 Top 3 steps to check if answer is fully supported by the metric:
1. Identify specific information in the context related to managing sepsis in a critical care unit.
2. Check if the answer directly quotes or paraphrases this informati

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [64]:
# The second query pass and user input
ground,rel = generate_ground_relevance_response(user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?", max_tokens = 370)

print(ground,end = "\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1. The answer accurately lists the common symptoms for appendicitis as described in the context.
2. The answer correctly states that appendicitis cannot be cured via medicine alone and requires surgical removal, as stated in the context.
3. The answer does not include any details or assumptions not explicitly supported by the context. For example, it does not mention any specific surgical procedures beyond what is mentioned in the context (surgical removal for appendicitis and possible drainage of an abscess).

Therefore, based on the assessment, I would assign a rating of 5 for the metric being fully followed by the answer.

 1. The answer accurately lists the common symptoms for appendicitis as described in the context.
2. The answer correctly states that appendicitis cannot be cured via medicine alone and requires surgical removal, as stated in the context.
3. The answer does not include any details or assumptions not explicitly supported by the context. For example, it does not me

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [65]:
# The third query pass and user input
ground,rel = generate_ground_relevance_response(user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?", max_tokens = 370)

print(ground,end = "\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Top 3 steps to check if the answer is fully supported by the metric:
1. Identify the specific information in the context related to effective treatments and causes of sudden patchy hair loss (alopecia areata).
2. Verify that all elements in the answer are directly derived from the provided context.
3. Cross-reference the information in the answer with the context to ensure accuracy and completeness.

Explanation: The answer is fully grounded in the given context as it mentions specific treatments (topical corticosteroids, immunosuppressants, Minoxidil, Diphencyprone, and Anthralin) for alopecia areata that are directly mentioned or discussed in the context. Additionally, the possible causes of sudden patchy hair loss (alopecia areata), including an autoimmune disorder, environmental triggers, stress, and viral infections, are also explicitly stated in the context.

Evaluation: The metric is fully followed as the answer only includes details, claims, or assumptions that are directly su

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [2]:
# The fourth query pass and user input
ground,rel = generate_ground_relevance_response(user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?", max_tokens = 370)

print(ground,end = "\n\n")
print(rel)

NameError: name 'generate_ground_relevance_response' is not defined

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [3]:
# The fifth query pass and user input
ground,rel = generate_ground_relevance_response(user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?", max_tokens = 370)

print(ground,end = "\n\n")
print(rel)

NameError: name 'generate_ground_relevance_response' is not defined

## Actionable Insights and Business Recommendations

Almost every query had high ratings, according to the metrics. Additionally, the RAG-based outputs were more descriptive and helpful compared to the ordinary LLM-based outputs thanks to the use of context from the manual.

If we want to improve the groundedness and relevance, we could always improve the chunking, chunk overlap, tokens, etc.

<font size=6 color='blue'>Power Ahead</font>
___